# Gold: sample classification manifest

Publishes a Gold `gold.manifest_rows` release, configured by `config/gold/manifests/<MANIFEST_NAME>.yaml` (mirrors `config/bronze/datasets/<dataset>.yaml`'s convention — one small YAML file per manifest, not per-manifest notebooks or hardcoded values). The default, `sample-v1`, spans `isic_2019` and `milk10k` (~100 images each), labeled on `malignancy` only, and exists to build and iterate on the Gold pipeline cheaply — not the final release. To build a different manifest, add a new YAML file under `config/gold/manifests/` and change `MANIFEST_NAME` below.

Sampling and split assignment are leakage-aware by construction: `data_platform.sampling.select_sample_and_splits` works in whole `silver.leakage_groups` groups, never individual images, so a group can never end up split across train/validation/test or partially sampled.

Each included dataset's own `20_*_silver_validate.ipynb` must have already run. A dataset with no accepted Silver rows yet for `malignancy` is skipped with a warning, not a failure.

Run `00_setup_storage_and_shared_tables.ipynb` before this notebook (creates `gold.manifest_rows` and the `gold.manifest_registry` overview view).


In [ ]:
%run ./_setup_env


In [ ]:
from data_platform.provenance import resolve_git_commit
from data_platform.sampling import select_sample_and_splits
from data_platform.dataset_layout import load_yaml_config
from data_platform.spark_io import (
    assert_no_cross_dataset_duplicate_checksums,
    build_gold_candidate_groups,
    print_gold_outputs,
    write_gold_manifest_rows,
)

# silver.*/gold.* are fixed, shared table names (not dataset-prefixed) -- inlined
# directly rather than via build_layout(), since build_layout is scoped to one
# dataset's config and these table names never actually vary by dataset_key.
SILVER_TABLES = {
    "image_inventory": "silver.image_inventory",
    "leakage_groups": "silver.leakage_groups",
    "rejected_records": "silver.rejected_records",
}
GOLD_TABLES = {"manifest_rows": "gold.manifest_rows"}

# Which manifest to build -- one small YAML file per manifest under
# config/gold/manifests/, mirroring config/bronze/datasets/<dataset>.yaml's existing
# convention. Change this to build a different manifest; to define a manifest
# that doesn't exist yet, add a new YAML file there rather than editing values
# in place here.
MANIFEST_NAME = "sample-v1"
MANIFEST_CONFIG = load_yaml_config(CONFIG_ROOT / "gold" / "manifests" / f"{MANIFEST_NAME}.yaml")

DATASET_KEYS = MANIFEST_CONFIG["dataset_keys"]
LABEL_COLUMN = MANIFEST_CONFIG["label_column"]
SAMPLE_SIZE_PER_DATASET = MANIFEST_CONFIG["sample_size_per_dataset"]
# sample_seed and split_seed are separate on purpose (see the config file's own
# comment): sample_seed controls which images are in the manifest at all,
# split_seed controls how they're divided into train/validation/test. Keep
# sample_seed fixed across manifests meant to be directly comparable (e.g. the
# same image selection under a different preprocessing_version).
SAMPLE_SEED = MANIFEST_CONFIG["sample_seed"]
SPLIT_RATIOS = tuple(MANIFEST_CONFIG["split_ratios"])
SPLIT_SEED = MANIFEST_CONFIG["split_seed"]
DATASET_VERSION = MANIFEST_CONFIG["dataset_version"]
PREPROCESSING_VERSION = MANIFEST_CONFIG["preprocessing_version"]

# The commit of the code writing this release, resolved up front so a missing commit fails
# before any work is done (see data_platform.provenance).
GIT_COMMIT = resolve_git_commit(REPO_ROOT)
print(f"git_commit: {GIT_COMMIT}")


## Build Candidate Leakage Groups

`build_gold_candidate_groups` aggregates each dataset's accepted, labeled Silver rows into one row per leakage-control group (cheap, metadata-only), collected to the driver as a plain list of dicts. A dataset with no accepted rows for `LABEL_COLUMN` yet is skipped with a warning.


In [ ]:
candidate_groups_by_dataset = {}
for dataset_key in DATASET_KEYS:
    groups = build_gold_candidate_groups(spark, SILVER_TABLES, dataset_key, LABEL_COLUMN)
    if not groups:
        print(f"{dataset_key}: no accepted rows with {LABEL_COLUMN} yet -- skipping (Silver not run for this dataset yet)")
        continue
    candidate_groups_by_dataset[dataset_key] = groups
    print(f"{dataset_key}: {len(groups)} candidate leakage groups")


## Check For Cross-Dataset Duplicate Images

`assert_no_cross_dataset_duplicate_checksums` raises if the same real image (by `source_checksum`, the sound cross-dataset identity signal -- `patient_id`/`lesion_id` are dataset-issued and not comparable across datasets) shows up in more than one of the datasets this manifest combines. A no-op when only one dataset is included. Run before sampling, not after: an undetected duplicate could otherwise land in different splits across its two copies -- real train/test leakage. See `docs/decisions/003-cross-dataset-leakage-not-checked.md`.


In [ ]:
assert_no_cross_dataset_duplicate_checksums(spark, SILVER_TABLES, list(candidate_groups_by_dataset))
print(f"No cross-dataset duplicate images found among {list(candidate_groups_by_dataset)}")


## Select Sample Groups And Assign Splits

`select_sample_and_splits` is pure Python and deterministic for a given `(groups, sample_size, sample_seed, split_seed)` — no Spark session needed for this step, and it's unit-tested locally (`tests/test_sampling.py`). `sample_seed` and `split_seed` are separate on purpose: keep `sample_seed` fixed across manifests meant to draw the exact same images (e.g. the same selection under a different `preprocessing_version`).


In [ ]:
split_by_group_by_dataset = {
    dataset_key: select_sample_and_splits(
        groups,
        sample_size=SAMPLE_SIZE_PER_DATASET,
        sample_seed=SAMPLE_SEED,
        split_seed=SPLIT_SEED,
        split_ratios=SPLIT_RATIOS,
    )
    for dataset_key, groups in candidate_groups_by_dataset.items()
}
for dataset_key, split_by_group in split_by_group_by_dataset.items():
    image_count = sum(
        g["image_count"] for g in candidate_groups_by_dataset[dataset_key] if g["group_id"] in split_by_group
    )
    print(f"{dataset_key}: {len(split_by_group)} groups selected, {image_count} images")


## Write gold.manifest_rows

`write_gold_manifest_rows` filters each dataset's accepted Silver rows down to the selected groups, attaches `split`, computes `manifest_row_hash`, and `MERGE INTO`s `gold.manifest_rows` keyed on `(dataset_version, dataset_key, image_id)` — rerunning this cell with the same inputs is idempotent, not duplicative. `gold.manifest_registry` (a view, see `00_setup_storage_and_shared_tables.ipynb`) picks up this release automatically — nothing extra to write there.


In [ ]:
for dataset_key, split_by_group in split_by_group_by_dataset.items():
    write_gold_manifest_rows(
        spark,
        SILVER_TABLES,
        GOLD_TABLES,
        dataset_key=dataset_key,
        label_column=LABEL_COLUMN,
        split_by_group=split_by_group,
        dataset_version=DATASET_VERSION,
        sample_seed=SAMPLE_SEED,
        split_seed=SPLIT_SEED,
        preprocessing_version=PREPROCESSING_VERSION,
        git_commit=GIT_COMMIT,
    )


## Review Outputs

`print_gold_outputs` shows rows per dataset, split distribution, and label distribution by split for this `DATASET_VERSION`, and raises if any leakage group ever ends up spanning more than one split (it shouldn't, by construction — this is a real invariant check, not just descriptive output).


In [ ]:
print_gold_outputs(
    spark, display, GOLD_TABLES, DATASET_VERSION, list(split_by_group_by_dataset), LABEL_COLUMN,
)


## Registry Entry For This Release

`gold.manifest_registry` is a view over every manifest ever written — this shows just the row for `DATASET_VERSION`, as a quick sanity check that this release looks right (datasets included, split counts, label values).


In [ ]:
display(spark.sql(f"SELECT * FROM gold.manifest_registry WHERE dataset_version = '{DATASET_VERSION}'"))
